# 25 — Prompt Governance and Responsible AI

## Scenario
Two developers submit prompts to production:
1. Alice writes a prompt to summarize internal meeting notes.
2. Bob writes a prompt to give users financial investment advice based on their bank accounts.

**The Problem:** If we treat all prompts equally, we either drown Alice in bureaucracy, or we let Bob deploy a massive compliance risk without oversight.

**The Solution:** We implement **Risk-Tiered Governance**. Prompts are assigned a Risk Tier (e.g., LOW, HIGH). LOW risk prompts are deployed automatically. HIGH risk prompts require explicit Human Review Board approval.

In [ ]:
from enum import Enum
from pydantic import BaseModel
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: The Governance Manifest

We require every prompt to be accompanied by a Governance Manifest, acting like a "System Card" for that specific behavior.

In [ ]:
class RiskTier(str, Enum):
    LOW = "LOW_RISK"
    MEDIUM = "MEDIUM_RISK"
    HIGH = "HIGH_RISK"

class GovernanceManifest(BaseModel):
    owner_email: str
    use_case: str
    risk_tier: RiskTier
    handles_pii: bool
    human_review_board_approval: bool = False

def governance_gate(manifest: GovernanceManifest) -> bool:
    print(f"\n--- Running Governance Gate for {manifest.use_case} ---")
    print(f"Owner: {manifest.owner_email} | Tier: {manifest.risk_tier.value} | PII: {manifest.handles_pii}")
    
    if manifest.risk_tier == RiskTier.LOW:
        print("✅ APPROVED: Low risk prompts can be auto-deployed.")
        return True
        
    elif manifest.risk_tier == RiskTier.HIGH:
        if manifest.human_review_board_approval:
            print("✅ APPROVED: High risk prompt has explicit Human Review Board approval.")
            return True
        else:
            print("❌ BLOCKED: High risk prompts require explicit Human Review Board approval before deployment.")
            return False
            
    return False


## Step 2: Testing the Governance Gate

Alice submits her internal summarizer. Bob submits his financial advisor.

In [ ]:
# Alice's Low Risk Prompt
alice_manifest = GovernanceManifest(
    owner_email="alice@company.com",
    use_case="Internal Meeting Summarizer",
    risk_tier=RiskTier.LOW,
    handles_pii=False
)
governance_gate(alice_manifest)

# Bob's High Risk Prompt (No Approval)
bob_manifest = GovernanceManifest(
    owner_email="bob@company.com",
    use_case="Customer-Facing Financial Advisor",
    risk_tier=RiskTier.HIGH,
    handles_pii=True
)
governance_gate(bob_manifest)

# Bob gets approval from Legal/Compliance and tries again
bob_manifest.human_review_board_approval = True
governance_gate(bob_manifest)


## Step 3: Technical Governance (SDK Safety Settings)

Governance isn't just about bureaucratic approvals. It's also about enforcing technical safeguards. The Google GenAI SDK allows you to explicitly set thresholds to block harmful content.

In [ ]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Write a highly offensive and threatening email.",
        config=types.GenerateContentConfig(
            safety_settings=[
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
                )
            ]
        )
    )
    print(response.text)
except Exception as e:
    print(f"\n[TECHNICAL GOVERNANCE ENFORCED] Request blocked by safety settings.\nError Details: {e}")


## Conclusion

Enterprise AI Governance requires both **Bureaucratic Controls** (Risk Tiers, Human Review Boards, Data Classification) and **Technical Controls** (SDK Safety Settings, PII Scrubbing).